In [5]:
import sys, time
from pathlib import Path

ROOT = Path('/Users/axel/axel/code/IKDH')
sys.path.insert(0, str(ROOT / 'build'))

import ikdh
from robodk.robolink import Robolink, ITEM_TYPE_ROBOT

# ── Target pose ───────────────────────────────────────────────────────────────
X, Y, Z    = 600.0, 0.0, 400.0  # mm
RX, RY, RZ = 0.0, 90.0, 0.0     # degrees (Rz·Ry·Rx)
INTERVAL   = 1.0                 # seconds between solutions

RDK = Robolink()

def _fanuc_to_rdk(q):
    out = list(q)
    out[2] = -q[1] - q[2]
    out[3] = -q[3]
    out[4] = -q[4]
    out[5] = -q[5]
    return out

def run(yaml, fanuc=False):
    robot     = ikdh.load_robot(yaml)
    robot_rdk = RDK.ItemList(ITEM_TYPE_ROBOT)[0]
    ee        = ikdh.pose_from_xyzrpw(X, Y, Z, RX, RY, RZ)
    solutions = ikdh.Solver(robot.dh, robot.limits).solve(ee)
    print(f"{robot.name}: {len(solutions)} solution(s) for ({X}, {Y}, {Z}) mm  Rx={RX} Ry={RY} Rz={RZ}")
    for i, q in enumerate(solutions):
        err   = ikdh.fk_error(ee, ikdh.forward_kin(robot.dh, q))
        q_out = _fanuc_to_rdk(q) if fanuc else list(q)
        print(f"  [{i+1:2d}/{len(solutions)}]  {' '.join(f'{v:7.2f}°' for v in q_out)}   err={err:.1e}")
        robot_rdk.setJoints(q_out)
        time.sleep(INTERVAL)
    print("Done.")

In [7]:
# ABB GoFa CRB 15000-5
time.sleep(2.0)
run(str(ROOT / 'robots/gofa5.yaml'))

ABB GoFa CRB 15000-5: 8 solution(s) for (600.0, 0.0, 400.0) mm  Rx=0.0 Ry=90.0 Rz=0.0
  [ 1/8]  -180.00° -144.70°   37.72°  180.00°   73.02°    0.00°   err=1.2e-29
  [ 2/8]  -180.00°  -22.72° -191.38° -180.00°  -34.10°   -0.00°   err=2.0e-23
  [ 3/8]     0.00°  124.17° -185.17°  180.00°  -61.00° -180.00°   err=2.7e-24
  [ 4/8]  -180.00° -124.17°   31.52°   -0.00°  -87.34° -180.00°   err=1.2e-23
  [ 5/8]     0.00°  144.70° -191.38°   -0.00°   46.67°    0.00°   err=1.0e-24
  [ 6/8]     0.00°   22.72°   37.72°    0.00°  -60.44°   -0.00°   err=7.3e-24
  [ 7/8]     0.00°    9.21°   31.52° -180.00°   40.73°  180.00°   err=1.4e-23
  [ 8/8]   180.00°   -9.21° -185.17°   -0.00°   14.38° -180.00°   err=7.5e-28
Done.


In [8]:
# FANUC CRX-10iA  (parallelogram coupling + sign inversions on J4/J5/J6)
time.sleep(2.0)
run(str(ROOT / 'robots/fanuc_crx_10ia.yaml'), fanuc=True)

Fanuc CRX-10iA: 8 solution(s) for (600.0, 0.0, 400.0) mm  Rx=0.0 Ry=90.0 Rz=0.0
  [ 1/8]  -161.20° -135.60°   99.27° -160.97°  -81.23°  176.99°   err=1.1e-25
  [ 2/8]   -18.80°  135.60° -279.27°  160.97°   81.23°    3.01°   err=1.0e-22
  [ 3/8]    18.80°  135.60° -279.27°   19.03°  -81.23°  176.99°   err=2.4e-22
  [ 4/8]   161.20° -135.60°   99.27°  -19.03°   81.23°    3.01°   err=1.5e-25
  [ 5/8]   -17.60°    3.55°  -37.54° -152.50°  -40.91°  -21.48°   err=3.0e-27
  [ 6/8]   162.40°   -3.55° -142.46°   27.50°  -40.91°  -21.48°   err=1.5e-21
  [ 7/8]  -162.40°   -3.55° -142.46°  152.50°   40.91° -158.52°   err=4.1e-29
  [ 8/8]    17.60°    3.55°  -37.54°  -27.50°   40.91° -158.52°   err=4.7e-27
Done.


In [9]:
# UR5e
time.sleep(2.0)
run(str(ROOT / 'robots/ur5e.yaml'))

UR5e: 8 solution(s) for (600.0, 0.0, 400.0) mm  Rx=0.0 Ry=90.0 Rz=0.0
  [ 1/8]   164.55°    6.80°  -87.96°   81.15°   74.55°  -90.00°   err=1.2e-26
  [ 2/8]    15.45° -114.82° -104.39°   39.21°   74.55°   90.00°   err=4.3e-25
  [ 3/8]   164.55°  -65.18°  104.39°  140.79°  -74.55°   90.00°   err=4.8e-29
  [ 4/8]    15.45° -103.28°  -87.96° -168.76°  -74.55°  -90.00°   err=6.8e-22
  [ 5/8]   164.55°  -76.72°   87.96°  -11.24°   74.55°  -90.00°   err=1.1e-27
  [ 6/8]    15.45°  146.71°  104.39°  -71.10°   74.55°   90.00°   err=2.1e-21
  [ 7/8]    15.45°  173.20°   87.96°   98.85°  -74.55°  -90.00°   err=1.1e-22
  [ 8/8]   164.55°   33.29° -104.39° -108.90°  -74.55°   90.00°   err=4.2e-22
Done.


In [10]:
# Doosan Robotics A0509 White
time.sleep(2.0)
run(str(ROOT / 'robots/doosan_robotics_a0509_white.yaml'))

Doosan Robotics A0509 White: 8 solution(s) for (600.0, 0.0, 400.0) mm  Rx=0.0 Ry=90.0 Rz=0.0
  [ 1/8]   180.00°  -19.59°  -92.98°   -0.00°   22.57°  -90.00°   err=2.1e-21
  [ 2/8]    -0.00°  106.04°  -92.98°    0.00°   76.94°   90.00°   err=8.3e-20
  [ 3/8]  -180.00° -106.04°   92.98°  180.00°   76.94°   90.00°   err=2.4e-21
  [ 4/8]    -0.00°  106.04°  -92.98° -180.00°  -76.94°  -90.00°   err=1.2e-22
  [ 5/8]     0.00°   19.59°   92.98°    0.00°  -22.57°   90.00°   err=5.0e-24
  [ 6/8]   180.00° -106.04°   92.98°    0.00°  -76.94°  -90.00°   err=4.2e-20
  [ 7/8]     0.00°   19.59°   92.98° -180.00°   22.57°  -90.00°   err=1.8e-27
  [ 8/8]   180.00°  -19.59°  -92.98°  180.00°  -22.57°   90.00°   err=1.5e-19
Done.
